In [1]:
import pandas as pd

df = pd.read_csv("UpdatedResumeDataSet.csv")

df.head()

In [2]:
df.tail()

In [3]:
df.shape

In [4]:
df.info

In [6]:
df.columns

In [8]:
df.describe(include='all')

In [9]:
df.isnull().sum()

In [11]:
df.duplicated().sum()

In [12]:
df.nunique()

In [14]:
df['Category'].value_counts()


In [19]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('UpdatedResumeDataSet.csv')

# Histogram of 'Age'
plt.hist(df['Category'].dropna(), bins=20, edgecolor='black')
plt.title('Distribution of candidates')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [20]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))

sns.countplot(
    data=df,
    y="Category",
    order=df["Category"].value_counts().index
)

plt.title("Resume Category Distribution")
plt.xlabel("Number of Resumes")
plt.ylabel("Category")

plt.show()

In [22]:
df["resume_char_count"] = (
    df["Resume"]
    .fillna("")
    .str.len()
)
df[["Resume", "resume_char_count"]].head()

In [24]:
df["resume_word_count"] = (
    df["Resume"]
    .fillna("")
    .str.split()
    .str.len()
)
df[["Resume", "resume_word_count"]].head()

In [25]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="resume_word_count",
    bins=30,
    kde=True
)

plt.title("Distribution of Resume Word Count")
plt.xlabel("Number of Words")
plt.ylabel("Number of Resumes")

plt.show()

In [26]:
short_resumes = df[df["resume_word_count"] < 50]

print("Number of very short resumes:", len(short_resumes))

short_resumes[["Category", "Resume", "resume_word_count"]].head(20)

In [27]:
long_resumes = df[df["resume_word_count"] > 2000]

print("Number of very long resumes:", len(long_resumes))

long_resumes[["Category", "resume_word_count"]].head(20)

In [28]:
plt.figure(figsize=(12, 8))

sns.boxplot(
    data=df,
    x="resume_word_count",
    y="Category"
)

plt.title("Resume Word Count by Job Category")
plt.xlabel("Number of Words")
plt.ylabel("Category")

plt.show()

In [30]:
import re
from collections import Counter

text = " ".join(
    df["Resume"]
    .fillna("")
    .astype(str)
    .str.lower()
)

text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

words = text.split()

word_counts = Counter(words)

word_counts.most_common(30)

In [31]:
import nltk

nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

filtered_words = [
    word
    for word in words
    if word not in stop_words
]
word_counts = Counter(filtered_words)

word_counts.most_common(30)

In [32]:
top_words = word_counts.most_common(20)

words_plot = [x[0] for x in top_words]
counts_plot = [x[1] for x in top_words]

plt.figure(figsize=(12, 7))

sns.barplot(
    x=counts_plot,
    y=words_plot
)

plt.title("Top 20 Most Frequent Words in Resumes")
plt.xlabel("Frequency")
plt.ylabel("Word")

plt.show()

In [41]:
df["char_count"] = df["Resume"].fillna("").str.len()

df["word_count"] = (
    df["Resume"]
    .fillna("")
    .str.split()
    .str.len()
)

df["sentence_count"] = (
    df["Resume"]
    .fillna("")
    .str.count(r"[.!?]")
)

In [43]:
plt.figure(figsize=(12, 8))

sns.boxplot(
    data=df,
    x="word_count",
    y="Category"
)

plt.title("Resume Word Count by Category")
plt.xlabel("Word Count")
plt.ylabel("Category")

plt.show()

In [44]:
avg_chars = (
    df.groupby("Category")["char_count"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 8))

avg_chars.plot(kind="barh")

plt.title("Average Resume Character Count by Category")
plt.xlabel("Average Character Count")
plt.ylabel("Category")

plt.show()

In [46]:
numeric_columns = [
    "word_count",
    "char_count",
    "sentence_count"
]

correlation = df[numeric_columns].corr()

print(correlation)

In [47]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix of Resume Features")

plt.show()

In [48]:
sns.pairplot(
    df[
        [
            "word_count",
            "char_count",
            "sentence_count",
            "Category"
        ]
    ],
    hue="Category",
    corner=True
)

plt.show()

In [55]:
df["word_count"] = df["Resume"].str.split().str.len()

df["char_count"] = df["Resume"].str.len()

df["sentence_count"] = (
    df["Resume"].str.count(r"[.!?]")
)

In [56]:
skills = [
    "python",
    "java",
    "javascript",
    "react",
    "node",
    "django",
    "fastapi",
    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "aws",
    "docker",
    "kubernetes",
    "tensorflow",
    "pytorch",
    "pandas",
    "numpy",
    "scikit-learn"
]

In [58]:
def count_skills(text):
    text = str(text).lower()

    return sum(
        skill in text
        for skill in skills
    )

df["skill_count"] = df["Resume"].apply(count_skills)

In [59]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df[
    ["word_count", "char_count", "skill_count"]
] = scaler.fit_transform(
    df[
        ["word_count", "char_count", "skill_count"]
    ]
)

In [60]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_features = scaler.fit_transform(
    df[["word_count", "char_count", "skill_count"]]
)

In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

resume_vectors = vectorizer.fit_transform(
    df["Resume"]
)

print(resume_vectors.shape)

In [62]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100)

reduced_vectors = svd.fit_transform(
    resume_vectors
)

print(reduced_vectors.shape)

# Complete Data Preprocessing & Feature Engineering Plan

## Objective

This notebook extends the existing Resume Intelligence EDA with the preprocessing,
feature engineering, feature selection, feature extraction, scaling, encoding,
dimensionality reduction, and validation steps required for the assignment.

The workflow uses the publicly available `UpdatedResumeDataSet.csv` dataset and
does not depend on proprietary candidate data.

> **Important:** The code is designed for a resume-text classification/matching
> workflow. One-Hot Encoding is demonstrated with a small synthetic categorical
> example because the current resume dataset does not contain a suitable structured
> categorical predictor such as Education or Location.


## 1. Why Data Preprocessing Is Important

Raw resume data can contain missing values, duplicates, empty records, inconsistent
text, unusual document lengths, and irrelevant tokens. These issues can affect
NLP representations and produce misleading model results.

The preprocessing strategy follows:

**Data Understanding → Data Quality → Cleaning → EDA → Feature Engineering →
Feature Extraction → Feature Selection → Scaling/Encoding → Dimensionality
Reduction → Validation**

Each step is justified by the properties of the data rather than applying
transformations automatically.


## 2. Python Libraries

- **pandas:** data loading, inspection, cleaning, grouping and transformation.
- **NumPy:** numerical calculations and array operations.
- **matplotlib:** basic visualization.
- **seaborn:** statistical visualization.
- **scikit-learn:** train/test splitting, TF-IDF, scaling, encoding, feature
  selection, cosine similarity and dimensionality reduction.
- **re:** controlled text normalization and skill matching.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")

DATA_PATH = "UpdatedResumeDataSet.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


## 3. Data Understanding and Quality Assessment

Before changing the dataset, inspect its structure. This establishes a reproducible
baseline and provides evidence for later preprocessing decisions.


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all").T)


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nTotal duplicate rows:")
print(df.duplicated().sum())

print("\nUnique values:")
print(df.nunique())


## 4. Missing Values

For text fields, missing values are handled explicitly. An empty string prevents
`NaN` values from propagating through string operations.

Completely empty resume records are separately identified because they do not
provide meaningful NLP information.


In [ ]:
print("Missing values BEFORE:")
print(df.isnull().sum())

df["Resume"] = df["Resume"].fillna("")

print("\nMissing values AFTER filling Resume:")
print(df.isnull().sum())

empty_resume_mask = df["Resume"].str.strip().eq("")
print("\nEmpty resume records:", empty_resume_mask.sum())


## 5. Duplicate Records

Duplicate records can make a model appear more accurate than it really is,
especially if identical records are split between training and testing data.
Duplicates are therefore removed before the train/test split.


In [ ]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

duplicates_after = df.duplicated().sum()

print("Duplicates before:", duplicates_before)
print("Duplicates after:", duplicates_after)
print("Rows remaining:", len(df))


## 6. Text Cleaning

Resume text contains useful technical terms, so cleaning should be conservative.
The cleaned text is stored separately so the original resume text remains available.


In [ ]:
def clean_resume_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_resume"] = df["Resume"].apply(clean_resume_text)

print("Original:")
print(df["Resume"].iloc[0][:300])

print("\nCleaned:")
print(df["clean_resume"].iloc[0][:300])


## 7. Feature Engineering

The following features describe resume structure:

- `word_count`
- `char_count`
- `sentence_count`
- `skill_count`

These features support EDA and can later contribute to a hybrid matching score.


In [ ]:
df["char_count"] = df["clean_resume"].str.len()
df["word_count"] = df["clean_resume"].str.split().str.len()
df["sentence_count"] = df["Resume"].str.count(r"[.!?]")

print(df[[
    "char_count", "word_count", "sentence_count"
]].describe())


In [ ]:
skills = [
    "python", "java", "javascript", "react", "node", "django", "fastapi",
    "sql", "mysql", "postgresql", "mongodb", "aws", "docker", "kubernetes",
    "tensorflow", "pytorch", "pandas", "numpy", "scikit-learn",
    "machine learning", "data science", "html", "css", "git"
]

def contains_skill(text, skill):
    pattern = r"(?<!\w)" + re.escape(skill.lower()) + r"(?!\w)"
    return bool(re.search(pattern, text.lower()))

def count_skills(text):
    return sum(contains_skill(text, skill) for skill in skills)

df["skill_count"] = df["clean_resume"].apply(count_skills)

print(df[[
    "Category", "word_count", "char_count",
    "sentence_count", "skill_count"
]].head())


## 8. Descriptive Statistics and Visualization

EDA is used to understand distributions, category balance, relationships between
features, and potential anomalies before modeling.


In [ ]:
print("Category distribution:")
display(df["Category"].value_counts())

print("\nResume feature statistics:")
display(
    df[["word_count", "char_count", "sentence_count", "skill_count"]].describe()
)


In [ ]:
plt.figure(figsize=(10, 6))
order = df["Category"].value_counts().index

sns.countplot(
    data=df,
    y="Category",
    order=order
)

plt.title("Resume Category Distribution")
plt.xlabel("Number of Resumes")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["word_count"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Resume Word Count Distribution")
axes[0].set_xlabel("Word Count")

sns.histplot(df["char_count"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Resume Character Count Distribution")
axes[1].set_xlabel("Character Count")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="word_count", y="Category")
plt.title("Resume Word Count by Category")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="word_count",
    y="char_count",
    hue="Category",
    alpha=0.7
)
plt.title("Word Count vs Character Count")
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = [
    "word_count", "char_count", "sentence_count", "skill_count"
]

plt.figure(figsize=(8, 6))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Matrix of Engineered Features")
plt.tight_layout()
plt.show()

sns.pairplot(
    df[numeric_cols + ["Category"]],
    hue="Category",
    corner=True
)
plt.show()


## 9. Outlier Analysis

A statistical outlier is not automatically an invalid resume. A very long resume
may be legitimate. Therefore, the IQR method identifies records for inspection
rather than automatically deleting them.


In [ ]:
Q1 = df["word_count"].quantile(0.25)
Q3 = df["word_count"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (df["word_count"] < lower_bound) |
    (df["word_count"] > upper_bound)
)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of statistical outliers:", outlier_mask.sum())

display(
    df.loc[outlier_mask, ["Category", "word_count", "char_count"]].head(10)
)


## 10. NumPy Numerical Analysis

NumPy complements pandas by providing numerical operations on arrays and columns.


In [ ]:
word_values = df["word_count"].to_numpy()

print("NumPy mean:", np.mean(word_values))
print("NumPy median:", np.median(word_values))
print("NumPy standard deviation:", np.std(word_values))


## 11. Train/Test Split and Leakage Prevention

Transformations that learn parameters from data should be fitted on the training
set only. This prevents information from the test set influencing the learned
representation.

The split is stratified by `Category` so category proportions are better preserved.


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Category"]
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining category proportions:")
print(train_df["Category"].value_counts(normalize=True))


## 12. TF-IDF Feature Extraction

TF-IDF is used as the lexical baseline because it is interpretable, efficient, and
well suited to measuring vocabulary overlap between resumes and job descriptions.

The vectorizer is fitted only on training text.


In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(train_df["clean_resume"])
X_test_tfidf = tfidf.transform(test_df["clean_resume"])

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

feature_names = tfidf.get_feature_names_out()

print("\nFirst 20 vocabulary terms:")
print(feature_names[:20])


In [ ]:
sample_query = [
    "Python Django developer with PostgreSQL Docker machine learning experience"
]

query_vector = tfidf.transform(sample_query)

similarities = cosine_similarity(
    query_vector,
    X_train_tfidf
)

print("Similarity matrix shape:", similarities.shape)
print("Top 5 similarity scores:",
      np.sort(similarities[0])[-5:][::-1])


## 13. Feature Selection

Feature extraction creates TF-IDF features. Feature selection chooses a subset
of useful existing features.

Chi-square selection is demonstrated because the dataset has categorical resume
labels. The selected feature count should be validated experimentally.


In [ ]:
k = min(1000, X_train_tfidf.shape[1])

selector = SelectKBest(
    score_func=chi2,
    k=k
)

X_train_selected = selector.fit_transform(
    X_train_tfidf,
    train_df["Category"]
)

X_test_selected = selector.transform(
    X_test_tfidf
)

print("Original training shape:", X_train_tfidf.shape)
print("Selected training shape:", X_train_selected.shape)
print("Original testing shape:", X_test_tfidf.shape)
print("Selected testing shape:", X_test_selected.shape)


## 14. Feature Scaling

Numerical engineered features can have different ranges. StandardScaler
standardizes them approximately to zero mean and unit variance, while MinMaxScaler
maps them into a bounded range.

The original columns are preserved.


In [ ]:
numeric_features = [
    "word_count", "char_count", "sentence_count", "skill_count"
]

standard_scaler = StandardScaler()

train_numeric_scaled = standard_scaler.fit_transform(
    train_df[numeric_features]
)

test_numeric_scaled = standard_scaler.transform(
    test_df[numeric_features]
)

scaled_train_df = pd.DataFrame(
    train_numeric_scaled,
    columns=[f"{c}_scaled" for c in numeric_features],
    index=train_df.index
)

display(scaled_train_df.head())


In [ ]:
minmax_scaler = MinMaxScaler()

train_numeric_minmax = minmax_scaler.fit_transform(
    train_df[numeric_features]
)

test_numeric_minmax = minmax_scaler.transform(
    test_df[numeric_features]
)

minmax_train_df = pd.DataFrame(
    train_numeric_minmax,
    columns=[f"{c}_minmax" for c in numeric_features],
    index=train_df.index
)

display(minmax_train_df.head())


## 15. One-Hot Encoding

The current resume dataset is primarily text-based and does not contain a clean
structured categorical predictor such as Education or Location. Therefore, a small
synthetic example demonstrates the technique without inventing fields in the real
dataset.

In a future structured version, OneHotEncoder can be applied to education level,
location, employment type, or experience level.


In [ ]:
demo_categories = pd.DataFrame({
    "Education": ["B.Com", "B.Tech", "MCA", "B.Com", "MBA"]
})

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_education = encoder.fit_transform(
    demo_categories[["Education"]]
)

encoded_education_df = pd.DataFrame(
    encoded_education,
    columns=encoder.get_feature_names_out(["Education"])
)

print("Original categorical data:")
display(demo_categories)

print("Encoded data:")
display(encoded_education_df)


## 16. Dimensionality Reduction with Truncated SVD

TF-IDF can generate thousands of sparse dimensions. Truncated SVD can compress
this representation into fewer components.

The number of components should be selected using explained-variance evidence and
downstream model performance rather than chosen arbitrarily.


In [ ]:
n_components = min(100, X_train_tfidf.shape[1] - 1)

svd = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

explained_variance = svd.explained_variance_ratio_.sum()

print("Original training shape:", X_train_tfidf.shape)
print("Reduced training shape:", X_train_svd.shape)
print("Original testing shape:", X_test_tfidf.shape)
print("Reduced testing shape:", X_test_svd.shape)
print("Explained variance ratio sum:", explained_variance)


In [ ]:
for components in [25, 50, 100]:
    if components < X_train_tfidf.shape[1]:
        temp_svd = TruncatedSVD(
            n_components=components,
            random_state=42
        )
        temp_svd.fit(X_train_tfidf)
        print(
            f"{components} components -> "
            f"{temp_svd.explained_variance_ratio_.sum():.4f} "
            "explained variance"
        )


## 17. Processing-Step Validation

Each transformation should be tested independently. Record the actual outputs
from this notebook as evidence in the report.

### Evidence to record

- Dataset shape before/after cleaning
- Missing values before/after
- Duplicate count before/after
- Empty resume count
- Outlier count and IQR bounds
- Engineered feature statistics
- TF-IDF matrix dimensions
- Example cosine similarity scores
- Number of selected features
- Scaling output/statistics
- One-hot encoded dimensions
- SVD dimensions and explained variance
- Train/test split sizes


In [ ]:
validation_summary = {
    "rows_after_cleaning": len(df),
    "missing_values_total": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "empty_resumes": int(df["clean_resume"].str.strip().eq("").sum()),
    "word_count_outliers": int(outlier_mask.sum()),
    "tfidf_train_rows": X_train_tfidf.shape[0],
    "tfidf_train_features": X_train_tfidf.shape[1],
    "selected_features": X_train_selected.shape[1],
    "svd_components": X_train_svd.shape[1],
    "svd_explained_variance": float(explained_variance)
}

for key, value in validation_summary.items():
    print(f"{key}: {value}")


## 18. Final Preprocessing Strategy

1. Load the public dataset with pandas.
2. Inspect structure, data types, missing values, duplicates and categories.
3. Fill missing resume text and identify completely empty records.
4. Remove exact duplicates before the train/test split.
5. Apply conservative text normalization.
6. Create word, character, sentence and skill-count features.
7. Perform univariate, bivariate and multivariate EDA.
8. Identify statistical outliers using IQR and inspect them before removal.
9. Split the dataset using a reproducible stratified train/test split.
10. Fit TF-IDF on training data and transform test data.
11. Use cosine similarity as the lexical matching baseline.
12. Apply chi-square feature selection as an experiment.
13. Scale numerical features with StandardScaler or MinMaxScaler when required.
14. Demonstrate One-Hot Encoding for future structured categorical features.
15. Use Truncated SVD when dimensionality reduction is justified.
16. Record actual outputs and observations for reproducibility.

The strategy intentionally starts with simple, interpretable techniques before
moving to more advanced representations. This creates a strong baseline for the
later hybrid resume-job matching system and eventual semantic embeddings.


## 19. Submission Evidence Checklist

- [ ] Dataset size recorded
- [ ] Missing values recorded before and after handling
- [ ] Duplicate count recorded before and after removal
- [ ] Empty records checked
- [ ] EDA plots included
- [ ] Outlier count and IQR bounds recorded
- [ ] Feature statistics recorded
- [ ] TF-IDF dimensions recorded
- [ ] Example similarity scores recorded
- [ ] Feature-selection dimensions recorded
- [ ] Scaling output verified
- [ ] One-hot encoding demonstrated
- [ ] SVD explained variance recorded
- [ ] Data-leakage prevention documented
- [ ] Final preprocessing decisions justified
